# xp-08 — Stronger models at the committed 0.30pp definition (pre-registered)

Pre-registered on 2026-08-16, **before any evaluation on the new test clients**.
xp-07 validated logistic regression at the adopted 0.30pp "below tier" definition
(rule 30/22, LR 80/92, base 18.6%, lift50 x4.96 vs x1.18). This experiment asks
whether a gradient-boosted model can beat that validated LR.

## Hypothesis
A gradient-boosted tree model (HistGB / XGB / LGBM) on the w05 feature set beats
the validated logistic regression at the committed 0.30pp cutoff, at BOTH
precision@10 and precision@50, on clients it never saw.

## Fixed setup (locked before the eval)
- Label: below_tier = (gap_label > 0.30).astype(int). The 0.30pp cutoff is the
  adopted definition from xp-07 (chosen there on train by a coverage rule).
- Split: GroupShuffleSplit, test_size 0.2, random_state 2027 - a fresh virgin
  seed, never used by any earlier decision (xp-01..06: 42-45; xp-07: 2026).
- Features: same w05 set (log_impressions_fw, ctr_fw, avg_pos_fw,
  pos_volatility_fw, engagement_rate_fw, log_sessions_fw, tier_ctr_gap;
  content_type, main_intent, position_tier).
- Models: LogisticRegression (baseline to beat), HistGB, XGB, LGBM - w05-style
  configs, logloss objective, no scale_pos_weight. No tuning on the eval set.
- Rule (for the record): has_volume x max(tier_ctr_gap, 0) x impressions_fw.
- Blend (xp-04 idea, folded in): alpha*rank(rule) + (1-alpha)*rank(model) with
  alpha chosen by client-grouped CV inside the train split only.

## Win criterion (one shot)
A model beats LR iff precision@10(model) > precision@10(LR) AND
precision@50(model) > precision@50(LR) AND precision@50(model) >= 2x the
virgin-test base rate. If a tree wins, it replaces LR as the validated model at
0.30pp; if not, LR stays. The rule stays the 0.1pp transparent baseline either way.

Guardrails: no tuning on the holdout; report everyone; audit the winner's top-50
(created-after-decision, content/client concentration, precision@k curve).


In [1]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Same w05 load (same rows, same order => same 120,258 pages).
data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']
raw_gap_label = data['gap_label'].copy()

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

print(f'Pages in data: {len(data):,}  Clients: {data["client_hash_id"].nunique():,}')

num_features = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
                'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_features = ['content_type', 'main_intent', 'position_tier']

def precision_at_k(score, y, k):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

out_dir = Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

from sklearn.model_selection import GroupShuffleSplit

# Fresh client holdout (seed 2026) - never used for any earlier decision
# (xp-01..06: 42-45; xp-07: 2026).
sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=2027)
train_idx, test_idx = next(sp.split(data, groups=data['client_hash_id']))
ntrain = data.iloc[train_idx].copy()
ntest = data.iloc[test_idx].copy()

print(f'Fresh train: {len(ntrain):,} pages from {ntrain["client_hash_id"].nunique()} clients')
print(f'Fresh test (virgin): {len(ntest):,} pages from {ntest["client_hash_id"].nunique()} clients')

# Committed cutoff from xp-07 (fixed): 0.30pp.
CUT = 0.30
cov = (raw_gap_label.loc[ntrain.index] > CUT).mean()
print(f'Below-tier coverage on fresh train at {CUT:.2f}pp: {cov:.1%}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Pages in data: 120,258  Clients: 47
Fresh train: 81,768 pages from 37 clients
Fresh test (virgin): 38,490 pages from 10 clients
Below-tier coverage on fresh train at 0.30pp: 19.6%


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

y_full = (raw_gap_label > CUT).astype(int)
y_tr = y_full.loc[ntrain.index]
y_te = y_full.loc[ntest.index]
base = y_te.mean()

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])
X_tr = pre.fit_transform(ntrain[num_features + cat_features])
X_te = pre.transform(ntest[num_features + cat_features])

models = {
    'lr': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'histgb': HistGradientBoostingClassifier(max_iter=200, early_stopping=True,
                validation_fraction=0.1, learning_rate=0.05, random_state=42),
}
try:
    from xgboost import XGBClassifier
    models['xgb'] = XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
    print('xgboost: available')
except Exception as e:
    print('xgboost: NOT available -', e)
try:
    from lightgbm import LGBMClassifier
    models['lgbm'] = LGBMClassifier(objective='binary', metric='binary_logloss',
                    n_estimators=200, num_leaves=31, learning_rate=0.05,
                    random_state=42, verbosity=-1)
    print('lightgbm: available')
except Exception as e:
    print('lightgbm: NOT available -', e)

has_volume = (ntest['impressions_fw'] >= 500).astype(int)
ctr_gap = ntest['tier_ctr_gap'].clip(lower=0)
rule_score = pd.Series((has_volume * ctr_gap * ntest['impressions_fw']).values, index=ntest.index)

print(f'Cutoff {CUT:.2f}pp | virgin-test base rate {base:.1%} (fresh seed 2027)')
scores = {'rule': rule_score}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    scores[name] = pd.Series(m.predict_proba(X_te)[:, 1], index=ntest.index)

rows = []
for name, s in scores.items():
    rows.append({
        'model': name,
        'p10': precision_at_k(s, y_te, 10),
        'p50': precision_at_k(s, y_te, 50),
        'p100': precision_at_k(s, y_te, 100),
        'p200': precision_at_k(s, y_te, 200),
        'lift50': precision_at_k(s, y_te, 50) / base,
    })
res = pd.DataFrame(rows)
res.to_csv(out_dir / 'xp08_models.csv', index=False)
print(res.round(4).to_string(index=False))
print()

lr_r = res[res['model'] == 'lr'].iloc[0]
print('Win check - model beats LR at BOTH cuts AND p@50 >= 2x base:')
for _, r in res.iterrows():
    if r['model'] in ('rule', 'lr'):
        continue
    ok = (r['p10'] > lr_r['p10']) and (r['p50'] > lr_r['p50']) and (r['p50'] >= 2 * base)
    print(f'  {r["model"]}: p10 {r["p10"]:.1%} > {lr_r["p10"]:.1%} | '
          f'p50 {r["p50"]:.1%} > {lr_r["p50"]:.1%} | lift50 x{r["lift50"]:.2f} -> {ok}')


xgboost: available
lightgbm: available
Cutoff 0.30pp | virgin-test base rate 14.2% (fresh seed 2027)
 model  p10  p50  p100  p200  lift50
  rule  0.4  0.2  0.15 0.125   1.405
    lr  1.0  1.0  1.00 1.000   7.025
histgb  1.0  1.0  1.00 1.000   7.025
   xgb  1.0  1.0  1.00 1.000   7.025
  lgbm  1.0  1.0  1.00 1.000   7.025

Win check - model beats LR at BOTH cuts AND p@50 >= 2x base:
  histgb: p10 100.0% > 100.0% | p50 100.0% > 100.0% | lift50 x7.03 -> False
  xgb: p10 100.0% > 100.0% | p50 100.0% > 100.0% | lift50 x7.03 -> False
  lgbm: p10 100.0% > 100.0% | p50 100.0% > 100.0% | lift50 x7.03 -> False


/Users/apple/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
from sklearn.model_selection import GroupKFold

def ranks(s):
    return s.rank(pct=True)

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

def cv_alpha(fit_predict, k=50, n_splits=5):
    # Client-grouped CV inside the train split only - never the eval set.
    gkf = GroupKFold(n_splits=n_splits)
    mean_p50 = {}
    for alpha in ALPHAS:
        vals = []
        for tr_fold, va_fold in gkf.split(ntrain, groups=ntrain['client_hash_id']):
            idx_tr = ntrain.index[tr_fold]
            idx_va = ntrain.index[va_fold]
            prob = fit_predict(idx_tr, idx_va)
            m_r = ranks(pd.Series(prob, index=idx_va))
            r_r = ranks(pd.Series(
                ((ntrain.loc[idx_va, 'impressions_fw'] >= 500).astype(int) *
                 ntrain.loc[idx_va, 'tier_ctr_gap'].clip(lower=0) *
                 ntrain.loc[idx_va, 'impressions_fw']).values, index=idx_va))
            blend = alpha * r_r + (1 - alpha) * m_r
            vals.append(precision_at_k(blend, y_tr.loc[idx_va], k))
        mean_p50[alpha] = float(np.mean(vals))
    return max(mean_p50, key=mean_p50.get), mean_p50

def lr_fit_predict(tr_idx, va_idx):
    p_tr = pre.fit_transform(ntrain.loc[tr_idx, num_features + cat_features])
    m = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    m.fit(p_tr, y_tr.loc[tr_idx])
    return m.predict_proba(pre.transform(ntrain.loc[va_idx, num_features + cat_features]))[:, 1]

def hgb_fit_predict(tr_idx, va_idx):
    p_tr = pre.fit_transform(ntrain.loc[tr_idx, num_features + cat_features])
    m = HistGradientBoostingClassifier(max_iter=200, early_stopping=True,
                validation_fraction=0.1, learning_rate=0.05, random_state=42)
    m.fit(p_tr, y_tr.loc[tr_idx])
    return m.predict_proba(pre.transform(ntrain.loc[va_idx, num_features + cat_features]))[:, 1]

print('Client-grouped CV on train (never the eval set) - best blend alpha:')
a_lr, cv_lr = cv_alpha(lr_fit_predict)
print(f'  rule+LR:      best alpha {a_lr} | mean p@50 by alpha: { {k: round(v, 4) for k, v in cv_lr.items()} }')
a_hgb, cv_hgb = cv_alpha(hgb_fit_predict)
print(f'  rule+HistGB:  best alpha {a_hgb} | mean p@50 by alpha: { {k: round(v, 4) for k, v in cv_hgb.items()} }')

print()
print('Blends evaluated once on the virgin test, at the chosen alpha:')
for bname, model_prob, alpha in [('blend_rule_lr', scores['lr'], a_lr),
                                  ('blend_rule_histgb', scores['histgb'], a_hgb)]:
    bs = alpha * ranks(rule_score) + (1 - alpha) * ranks(model_prob)
    bp10 = precision_at_k(bs, y_te, 10)
    bp50 = precision_at_k(bs, y_te, 50)
    print(f'  {bname} (alpha={alpha}): p@10 {bp10:.1%} | p@50 {bp50:.1%}  '
          f'(LR baseline {lr_r["p10"]:.1%}/{lr_r["p50"]:.1%})')


Client-grouped CV on train (never the eval set) - best blend alpha:
  rule+LR:      best alpha 0.25 | mean p@50 by alpha: {0.0: 0.864, 0.25: 0.868, 0.5: 0.768, 0.75: 0.704, 1.0: 0.184}
  rule+HistGB:  best alpha 0.5 | mean p@50 by alpha: {0.0: 0.864, 0.25: 0.932, 0.5: 0.948, 0.75: 0.896, 1.0: 0.184}

Blends evaluated once on the virgin test, at the chosen alpha:
  blend_rule_lr (alpha=0.25): p@10 100.0% | p@50 90.0%  (LR baseline 100.0%/100.0%)
  blend_rule_histgb (alpha=0.5): p@10 100.0% | p@50 100.0%  (LR baseline 100.0%/100.0%)


In [4]:
best = res[res['model'] != 'rule'].sort_values('p50', ascending=False).iloc[0]
best_model = best['model']
print(f'Auditing the top-50 of the best non-rule method: {best_model} (p@50 {best["p50"]:.1%})')

created = con.sql(
    f"SELECT content_hash_id, content_created_date FROM read_parquet('{REL}/dim_content.parquet')"
).df()
ntest_a = ntest.copy()
ntest_a['created_date'] = ntest_a['content_hash_id'].map(
    created.set_index('content_hash_id')['content_created_date']
)
ntest_a['days_since_created'] = (
    pd.to_datetime('2026-03-01') - pd.to_datetime(ntest_a['created_date'])
).dt.days
created_after_decision = (ntest_a['days_since_created'] < 0).astype(int)

top50 = ntest_a.loc[scores[best_model].nlargest(50).index]
print(f'  created after decision date: {created_after_decision.loc[top50.index].mean():.1%}')
print(f'  content_type:\n{top50["content_type"].value_counts(normalize=True).round(2).to_string()}')
print(f'  position_tier:\n{top50["position_tier"].value_counts(normalize=True).round(2).to_string()}')
print(f'  clients represented: {top50["client_hash_id"].nunique()} of {ntest["client_hash_id"].nunique()}')
print(f'  median impressions_fw: {top50["impressions_fw"].median():,.0f} | '
      f'median sessions_fw: {top50["sessions_fw"].median():,.0f}')

print()
print('Precision@k for every method (guardrail: not a single lucky cutoff):')
for k in [10, 50, 100, 200]:
    parts = [f'{name} {precision_at_k(s, y_te, k):.1%}' for name, s in scores.items()]
    print(f'  k={k:>3}: ' + ' | '.join(parts))


Auditing the top-50 of the best non-rule method: lr (p@50 100.0%)
  created after decision date: 0.0%
  content_type:
content_type
comparison article    0.98
keyword article       0.02
  position_tier:
position_tier
page_1    1.0
  clients represented: 3 of 10
  median impressions_fw: 130 | median sessions_fw: 0

Precision@k for every method (guardrail: not a single lucky cutoff):
  k= 10: rule 40.0% | lr 100.0% | histgb 100.0% | xgb 100.0% | lgbm 100.0%
  k= 50: rule 20.0% | lr 100.0% | histgb 100.0% | xgb 100.0% | lgbm 100.0%
  k=100: rule 15.0% | lr 100.0% | histgb 100.0% | xgb 100.0% | lgbm 100.0%
  k=200: rule 12.5% | lr 100.0% | histgb 100.0% | xgb 100.0% | lgbm 100.0%


## Verdict (fill after running)

- [ ] Report the fresh-seed (2027) virgin-test base rate and the p@10/p@50 table for rule, LR, HistGB, XGB, LGBM, and the blends.
- [ ] Does any gradient-boosted model beat LR at BOTH p@10 and p@50 and clear ~2x the base rate? (win criterion)
- [ ] Audit gate: top-50 of the best model - created-after-decision %, content/client concentration, precision@k curve.
- [ ] If a tree wins and the audit is clean: it replaces LR as the validated model at 0.30pp. If not: LR stays.
